In [ ]:
import os
import cv2
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, Concatenate, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler, ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def iou_coef(y_true, y_pred, smooth=1):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


In [ ]:
# Function to split images into tiles
def split_image_into_tiles(image_path, mask_path, tile_size, size):
    img = tifffile.imread(image_path)
    mask = tifffile.imread(mask_path)
    mask = mask[:, :, 0] if len(mask.shape) == 3 else mask

    tiles_img, tiles_mask = [], []
    for x in range(0, img.shape[1], tile_size):
        for y in range(0, img.shape[0], tile_size):
            tile_img = img[y:y+tile_size, x:x+tile_size, :]
            tile_mask = mask[y:y+tile_size, x:x+tile_size]

            tile_img = cv2.resize(tile_img, (size, size))
            tile_mask = cv2.resize(tile_mask, (size, size))
            tile_mask = (tile_mask > 0).astype(np.uint8)

            tiles_img.append(tile_img)
            tiles_mask.append(tile_mask)

    return np.array(tiles_img), np.array(tiles_mask)

# Load dataset
def load_data(image_dir, mask_dir, tile_size=256, size=256):
    images, masks = [], []
    image_filenames = sorted(os.listdir(image_dir))
    mask_filenames = sorted(os.listdir(mask_dir))

    for image_filename in image_filenames:
        if image_filename.endswith(".TIF"):
            mask_filename = image_filename.replace(".TIF", "_mask.TIF")
            if mask_filename in mask_filenames:
                img_path = os.path.join(image_dir, image_filename)
                mask_path = os.path.join(mask_dir, mask_filename)
                img, mask = split_image_into_tiles(img_path, mask_path, tile_size, size)
                images.extend(img)
                masks.extend(mask)
    return np.array(images), np.array(masks)

# Paths
image_dir = "../../datasets/images"
mask_dir = "../../datasets/masks"
size = 256

# Load data
tiles_img, tiles_mask = load_data(image_dir, mask_dir, tile_size=size, size=size)

# Split sets
X_train, X_test, y_train, y_test = train_test_split(tiles_img, tiles_mask, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

# Reshape masks
y_train = y_train[..., np.newaxis]
y_val = y_val[..., np.newaxis]
y_test = y_test[..., np.newaxis]


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, Concatenate
from tensorflow.keras.optimizers import Adam

def resnet50_unet_model(input_size=(256, 256, 3), freeze_encoder=True):
    resnet50_base = ResNet50(weights='imagenet', include_top=False, input_shape=input_size)
    encoder_output = resnet50_base.get_layer('conv5_block3_out').output

    if freeze_encoder:
        for layer in resnet50_base.layers:
            layer.trainable = False

    # Decoder
    x = Conv2DTranspose(512, (2,2), strides=(2,2), padding="same")(encoder_output)
    x = Concatenate()([x, resnet50_base.get_layer('conv4_block6_out').output])
    x = Conv2D(512, (3,3), padding="same", activation="relu")(x)

    x = Conv2DTranspose(256, (2,2), strides=(2,2), padding="same")(x)
    x = Concatenate()([x, resnet50_base.get_layer('conv3_block4_out').output])
    x = Conv2D(256, (3,3), padding="same", activation="relu")(x)

    x = Conv2DTranspose(128, (2,2), strides=(2,2), padding="same")(x)
    x = Concatenate()([x, resnet50_base.get_layer('conv2_block3_out').output])
    x = Conv2D(128, (3,3), padding="same", activation="relu")(x)

    x = Conv2DTranspose(64, (2,2), strides=(2,2), padding="same")(x)
    x = Concatenate()([x, resnet50_base.get_layer('conv1_conv').output])
    x = Conv2D(64, (3,3), padding="same", activation="relu")(x)

    # Extra upsampling to reach 256x256
    x = Conv2DTranspose(32, (2,2), strides=(2,2), padding="same")(x)
    x = Conv2D(32, (3,3), padding="same", activation="relu")(x)

    output = Conv2D(1, (1,1), activation="sigmoid")(x)

    model = Model(inputs=resnet50_base.input, outputs=output)
    model.compile(optimizer=Adam(1e-4),
                  loss="binary_crossentropy",
                  metrics=["accuracy", dice_coef, iou_coef])
    return model

# Usage
size = 256
model = resnet50_unet_model(input_size=(size, size, 3), freeze_encoder=True)
model.summary()


In [ ]:
# LR schedule
def lr_schedule(epoch):
    initial_lr = 1e-4
    decay = 0.9
    return initial_lr * (decay ** (epoch // 10))

lr_scheduler = LearningRateScheduler(lr_schedule)

# Data augmentation
datagen = ImageDataGenerator(rescale=1./255,
                             shear_range=0.2,
                             zoom_range=0.2,
                             horizontal_flip=True,
                             rotation_range=20,
                             width_shift_range=0.2,
                             height_shift_range=0.2,
                             brightness_range=[0.8, 1.2])

# Callbacks
checkpointer = ModelCheckpoint("best_unet_resnet50.h5", monitor="val_dice_coef", mode="max",
                               save_best_only=True, verbose=1)
earlyStopping = EarlyStopping(monitor="val_dice_coef", patience=5, mode="max", verbose=1)


In [ ]:
history = model.fit(datagen.flow(X_train, y_train, batch_size=32),
                    validation_data=(X_val/255.0, y_val),
                    epochs=50,
                    callbacks=[lr_scheduler, earlyStopping, checkpointer])


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample



def bootstrap_confidence_interval(y_true, y_pred, metric_fn, n_bootstraps=100, alpha=0.95):
    """Bootstrap CI + std for a given metric"""
    stats = []
    n = len(y_true)
    for _ in range(n_bootstraps):
        indices = np.random.randint(0, n, n)
        if metric_fn.__name__ == "roc_auc_score":  # ROC-AUC requires probs
            stat = metric_fn(y_true[indices], y_pred[indices])
        else:  # Binary metrics
            stat = metric_fn(y_true[indices], y_pred[indices])
        stats.append(stat)
    
    stats = np.array(stats)
    mean_val = np.mean(stats)
    std_val  = np.std(stats)
    lower = np.percentile(stats, ((1 - alpha) / 2) * 100)
    upper = np.percentile(stats, (alpha + (1 - alpha) / 2) * 100)
    
    return mean_val, std_val, (lower, upper)


# --- Evaluate model ---
loss, acc, dice, iou = model.evaluate(X_test/255.0, y_test)
print(f"Test Loss: {loss:.4f}, Accuracy: {acc:.4f}, Dice: {dice:.4f}, IoU: {iou:.4f}")

# --- Predictions ---
y_pred = model.predict(X_test/255.0)
y_pred_bin = (y_pred > 0.5).astype(np.uint8)

# Flatten
y_true_flat = y_test.flatten()
y_pred_flat = y_pred_bin.flatten()
y_pred_probs = y_pred.flatten()

# --- Metrics ---
metrics = {
    "Accuracy": lambda yt, yp: np.mean(yt == yp),
    "Precision": lambda yt, yp: precision_score(yt, yp),
    "Recall": lambda yt, yp: recall_score(yt, yp),
    "F1-score": lambda yt, yp: f1_score(yt, yp),
    "ROC-AUC": lambda yt, yp: roc_auc_score(yt, yp),
    "Dice": lambda yt, yp: (2*np.sum(yt*yp))/(np.sum(yt)+np.sum(yp)+1e-7),
    "IoU": lambda yt, yp: np.sum(yt*yp)/(np.sum(yt)+np.sum(yp)-np.sum(yt*yp)+1e-7)
}

print("\n📊 Metrics with 95% Confidence Intervals:")
for name, fn in metrics.items():
    if name == "ROC-AUC":
        mean_val, std_val, (low, high) = bootstrap_confidence_interval(y_true_flat, y_pred_probs, fn)
    else:
        mean_val, std_val, (low, high) = bootstrap_confidence_interval(y_true_flat, y_pred_flat, fn)
    print(f"{name}: {mean_val:.4f} ± {std_val:.4f}  (95% CI: {low:.4f} – {high:.4f})")


In [ ]:
y_pred = model.predict(X_test)
y_pred_bin = (y_pred > 0.5).astype(int)

y_true_flat = y_test.flatten()
y_pred_flat = y_pred_bin.flatten()

cm = confusion_matrix(y_true_flat, y_pred_flat)
print("Confusion Matrix:\n", cm)
print(classification_report(y_true_flat, y_pred_flat))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Forest","Forest"],
            yticklabels=["Non-Forest","Forest"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# Plot training history for loss, accuracy, dice, iou
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')

plt.xlabel('Epoch')
plt.ylabel('Metrics')
plt.title('Resnet_U-Net Segmentation Training History')
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Epochs: 1 to 27 (early stopping at 27)
epochs = range(1, 28)

# Training metrics
train_loss = [
    0.8038, 0.4354, 0.3695, 0.3409, 0.3317, 0.3236, 0.3162, 0.3050, 0.3037, 0.3022,
    0.3047, 0.2919, 0.2913, 0.2922, 0.2978, 0.2914, 0.2915, 0.2929, 0.2915, 0.2853,
    0.2786, 0.2834, 0.2833, 0.2867, 0.2801, 0.2819, 0.2762
]

train_accuracy = [
    0.7276, 0.7707, 0.8129, 0.8384, 0.8422, 0.8489, 0.8545, 0.8609, 0.8615, 0.8616,
    0.8588, 0.8681, 0.8680, 0.8677, 0.8631, 0.8672, 0.8692, 0.8652, 0.8665, 0.8706,
    0.8758, 0.8715, 0.8712, 0.8688, 0.8733, 0.8717, 0.8746
]

train_dice = [
    0.6957, 0.7532, 0.7847, 0.7938, 0.8071, 0.8107, 0.8143, 0.8225, 0.8234, 0.8235,
    0.8210, 0.8300, 0.8307, 0.8317, 0.8277, 0.8311, 0.8299, 0.8304, 0.8344, 0.8290,
    0.8388, 0.8331, 0.8354, 0.8360, 0.8362, 0.8372, 0.8392
]

train_iou = [
    0.5398, 0.6072, 0.6483, 0.6642, 0.6789, 0.6839, 0.6892, 0.7015, 0.7020, 0.7021,
    0.7000, 0.7109, 0.7121, 0.7138, 0.7082, 0.7136, 0.7114, 0.7133, 0.7182, 0.7114,
    0.7240, 0.7168, 0.7191, 0.7204, 0.7211, 0.7219, 0.7252
]

# Validation metrics
val_loss = [
    0.4534, 0.3617, 0.2979, 0.3126, 0.2763, 0.2690, 0.2724, 0.2595, 0.2555, 0.2572,
    0.2524, 0.2496, 0.2435, 0.2451, 0.2660, 0.2603, 0.2424, 0.2479, 0.2550, 0.2391,
    0.2444, 0.2380, 0.2390, 0.2434, 0.2579, 0.2395, 0.2423
]

val_accuracy = [
    0.7790, 0.8096, 0.8662, 0.8519, 0.8721, 0.8852, 0.8873, 0.8904, 0.8937, 0.8869,
    0.8942, 0.8939, 0.8952, 0.8927, 0.8923, 0.8906, 0.9015, 0.8914, 0.8940, 0.9010,
    0.8974, 0.9002, 0.9019, 0.8945, 0.8930, 0.9009, 0.8961
]

val_dice = [
    0.7284, 0.7708, 0.8058, 0.8185, 0.8273, 0.8302, 0.8218, 0.8327, 0.8347, 0.8325,
    0.8412, 0.8368, 0.8474, 0.8393, 0.8337, 0.8492, 0.8411, 0.8394, 0.8336, 0.8499,
    0.8530, 0.8561, 0.8468, 0.8398, 0.8472, 0.8478, 0.8417
]

val_iou = [
    0.5755, 0.6300, 0.6773, 0.6959, 0.7075, 0.7119, 0.6997, 0.7153, 0.7183, 0.7150,
    0.7279, 0.7213, 0.7368, 0.7247, 0.7173, 0.7404, 0.7278, 0.7249, 0.7165, 0.7409,
    0.7458, 0.7503, 0.7360, 0.7255, 0.7373, 0.7378, 0.7283
]

In [ ]:
plt.figure(figsize=(16, 12))

# Plot 1: Loss
plt.subplot(2, 2, 1)
plt.plot(epochs, train_loss, 'bo-', label='Training Loss', linewidth=2)
plt.plot(epochs, val_loss, 'r-o', label='Validation Loss', linewidth=2)
plt.title('Training and Validation Loss', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Accuracy
plt.subplot(2, 2, 2)
plt.plot(epochs, train_accuracy, 'bo-', label='Training Accuracy', linewidth=2)
plt.plot(epochs, val_accuracy, 'r-o', label='Validation Accuracy', linewidth=2)
plt.title('Training and Validation Accuracy', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 3: Dice Coefficient
plt.subplot(2, 2, 3)
plt.plot(epochs, train_dice, 'bo-', label='Training Dice Coefficient', linewidth=2)
plt.plot(epochs, val_dice, 'r-o', label='Validation Dice Coefficient', linewidth=2)
plt.title('Training and Validation Dice Coefficient', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Dice Coefficient')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 4: IoU
plt.subplot(2, 2, 4)
plt.plot(epochs, train_iou, 'bo-', label='Training IoU', linewidth=2)
plt.plot(epochs, val_iou, 'r-o', label='Validation IoU', linewidth=2)
plt.title('Training and Validation IoU', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('IoU')
plt.legend()
plt.grid(True, alpha=0.3)

# Adjust layout and show
plt.tight_layout()
plt.show()